<center>

# **Data Call**

In [1]:
from datasets import load_dataset

ds = load_dataset("papluca/language-identification")

README.md: 0.00B [00:00, ?B/s]

c:\Users\ZBook\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ZBook\.cache\huggingface\hub\datasets--papluca--language-identification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv:   0%|          | 0.00/12.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 70000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 10000
    })
})
{'labels': 'pt', 'text': 'os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.'}


### Convert it to pandas

In [8]:
df = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()


print(df.sample(5))

      labels                                               text
21241     ru  С большим пляжем , одним из лучших французских...
66596     de  Der Tragegurt war nur an einer Seite befestigt...
48805     fr  Paquet arrivé ouvert et livre abîmé sur la cou...
38386     el  Ρισκάρουμε την ύβρις , δεδομένης της ανιαρή εθ...
66495     en  Continues the series. Another great installmen...


In [10]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

df.sample(5)

,labels,text
20479,th,ผู้ซื้อ ที่ ฉลาด คือ คน ที่ จดจำ พนักงาน inhouse ที่ มี ความสามารถ
18829,en,Didn’t work as expected
31929,pl,Zapasy USA spadają w miarę rozmów greckich z tarcicą; jabłko spada
14269,th,Mus ?? e camargais ที่ albaron ให้ เหลือบดู วิถีชีวิต แบบ ดั้งเดิม ของ พื้นที่ และ กำหนด คุณ ออก บน 3.5 กม. ( 2 - ไมล์ ) เดินผ่าน ทาง marshland และ the parc ornithologique ที่ pont de gau อนุญาต ให้ คุณ เพื่อ ดู จำนวน สายพันธุ์ นก โดย ไม่ต้อง ค้าง นาน เข้า ภายใน
33488,zh,作者有一种不可一世的态度。。但是废话不少。。真的讲到关键处么。。又那样带过的赶脚。。


In [15]:
# number of unique classes
num_classes = df["labels"].nunique()

print("Number of classes:", num_classes)

print("=" * 50)

# count samples in each class
class_counts = df["labels"].value_counts()

print(class_counts)

Number of classes: 20
labels
pt    3500
bg    3500
en    3500
vi    3500
fr    3500
nl    3500
el    3500
de    3500
hi    3500
it    3500
ar    3500
es    3500
tr    3500
sw    3500
ur    3500
pl    3500
ru    3500
th    3500
zh    3500
ja    3500
Name: count, dtype: int64


<center>

# **preprocessing**

In [18]:
import re
import string
import emoji

class TextPreprocessor:
    def __init__(self):
        self.punctuations = string.punctuation

    def remove_numbers(self, text):
        return re.sub(r'\d+', '', text)

    def remove_links(self, text):
        return re.sub(r'http\S+|www\S+', '', text)

    def remove_emojis(self, text):
        return emoji.replace_emoji(text, replace='')

    def remove_punctuations(self, text):
        return text.translate(str.maketrans('', '', self.punctuations))

    def clean_text(self, text):
        text = text.lower()
        text = self.remove_numbers(text)
        text = self.remove_links(text)
        text = self.remove_emojis(text)
        text = self.remove_punctuations(text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text


### Applay on the data

In [21]:
prep = TextPreprocessor()

df["clean_text"] = df.apply(lambda row: prep.clean_text(row["text"]), axis=1)

print(df[["text", "labels", "clean_text"]].head())


                                                                                                                                                                text  \
0  os chefes de defesa da estónia, letónia, lituânia, alemanha, itália, espanha e eslováquia assinarão o acordo para fornecer pessoal e financiamento para o centro.   
1                                размерът на хоризонталната мрежа може да бъде по реда на няколко километра ( km ) за на симулация до около 100 km за на симулация .   
2                                                   很好，以前从不去评价，不知道浪费了多少积分，现在知道积分可以换钱，就要好好评价了，后来我就把这段话复制走了，既能赚积分，还省事，走到哪复制到哪，最重要的是，不用认真的评论了，不用想还差多少字，直接发出就可以了，推荐给大家！！   
3                      สำหรับ ของเก่า ที่ จริงจัง ลอง   honeychurch   ของเก่า ที่ ไม่   29   สำหรับ เฟอร์นิเจอร์ และ เงิน ไท ร้อง บริษัท ที่   122   สำหรับ ลาย คราม   
4                                                                                                                                             Он увеличил давлен

In [22]:
df.sample(10)

,labels,text,clean_text
36881,hi,पैसे जेनरेट करने के अन ् य तरीकों से वकालत कर रहे हैं .,पैसे जेनरेट करने के अन ् य तरीकों से वकालत कर रहे हैं
42840,ar,نحن ما زلنا نستخدم تلك الاشياء,نحن ما زلنا نستخدم تلك الاشياء
69255,ar,وستكون اختبارات الفحص المصممة للمرضى الذين يعانون من مشاكل اكثر حدة ( 6 مشروبات ) اقل حساسية في تحديد المرضى الذين يعانون من مشاكل اقل حدة ( 3 مشروبات ) .,وستكون اختبارات الفحص المصممة للمرضى الذين يعانون من مشاكل اكثر حدة مشروبات اقل حساسية في تحديد المرضى الذين يعانون من مشاكل اقل حدة مشروبات
32601,fr,L’effet d’ouverture est très surprenant et puissant. Il s’est éjecté et a faillit blesser la personne assise en face. D’en plus il est très difficile à refermer et toutes les personne qui ont essayé se sont blessé à la main. Dommage qu’il n’y a pas de manuel qui explique le fonctionnement et comment refermer sans risque.,l’effet d’ouverture est très surprenant et puissant il s’est éjecté et a faillit blesser la personne assise en face d’en plus il est très difficile à refermer et toutes les personne qui ont essayé se sont blessé à la main dommage qu’il n’y a pas de manuel qui explique le fonctionnement et comment refermer sans risque
7868,en,"I guess my review of realistic and others is very different, but truly I don't like this one.",i guess my review of realistic and others is very different but truly i dont like this one
26933,ar,انا محتار بشان ما اقول بشان موضوع ديني .,انا محتار بشان ما اقول بشان موضوع ديني
39947,th,' ไม่น่าเลย,ไม่น่าเลย
41426,zh,就是简单地将英文的review翻译一下，而且感觉可能是译者的免疫学背景不是很强，一些翻译的语句读起来不好理解。 由于只是简单的review翻译，所以全文段落排版啥的也很粗糙。 建议想想连接这方面的还是看英文原著或者review吧，这本书这不适合。,就是简单地将英文的review翻译一下，而且感觉可能是译者的免疫学背景不是很强，一些翻译的语句读起来不好理解。 由于只是简单的review翻译，所以全文段落排版啥的也很粗糙。 建议想想连接这方面的还是看英文原著或者review吧，这本书这不适合。
48280,es,"Son unos auriculares cómodos en general, se quedan bien colocados en la oreja aunque el mando del volumen es un poco grande y molesta al correr, yo intento sujetarlo con el borde de la camiseta pero resulta un poco incómodo. Para mí éste es es peor aspecto de los auriculares. Por otro lado, por el precio que tienen, ofrecen buena calidad de sonido y no te aíslan del ruido exterior cuando vas corriendo, algo importante para permanecer alerta.",son unos auriculares cómodos en general se quedan bien colocados en la oreja aunque el mando del volumen es un poco grande y molesta al correr yo intento sujetarlo con el borde de la camiseta pero resulta un poco incómodo para mí éste es es peor aspecto de los auriculares por otro lado por el precio que tienen ofrecen buena calidad de sonido y no te aíslan del ruido exterior cuando vas corriendo algo importante para permanecer alerta
15680,el,Σκοτώνοντας τα γόνατά μου,σκοτώνοντας τα γόνατά μου


<center>

# **TF‑IDF Vectorization**

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfLangVectorizer:
    def __init__(self, max_features=5000, ngram_range=(2,5)):

        self.vectorizer = TfidfVectorizer(
            analyzer='char',
            max_features=max_features,
            ngram_range=ngram_range
        )
        self.fitted = False

    def fit(self, texts):
        self.vectorizer.fit(texts)
        self.fitted = True

    def transform(self, texts):
        if not self.fitted:
            raise ValueError("Vectorizer not fitted yet. Call fit() first.")
        return self.vectorizer.transform(texts)

    def fit_transform(self, texts):
        X = self.vectorizer.fit_transform(texts)
        self.fitted = True
        return X


In [24]:
prep = TextPreprocessor()
df["clean_text"] = df["text"].apply(prep.clean_text)

tfidf = TfidfLangVectorizer(max_features=5000, ngram_range=(2,5))
X = tfidf.fit_transform(df["clean_text"])
y = df["labels"]

print("Shape of TF-IDF matrix:", X.shape)


Shape of TF-IDF matrix: (70000, 5000)


In [26]:
vocab = tfidf.vectorizer.vocabulary_
print("Size:", len(vocab))

# شوف بعض الـ n-grams
print(list(vocab.keys())[:200])


Size: 5000
['os', 's ', ' c', 'ch', 'he', 'ef', 'fe', 'es', ' d', 'de', 'e ', 'sa', 'a ', 'da', ' e', 'st', 'ón', 'ni', 'ia', ' l', 'le', 'et', 'li', 'it', 'tu', 'ân', ' a', 'al', 'em', 'ma', 'an', 'nh', 'ha', ' i', 'tá', 'sp', 'pa', 'sl', 'lo', 'ov', 'qu', 'ui', 'as', 'ss', 'si', 'in', 'na', 'ar', 'ão', 'o ', ' o', 'ac', 'co', 'or', 'rd', 'do', ' p', 'ra', ' f', 'fo', 'rn', 'ne', 'ec', 'ce', 'er', 'r ', 'pe', 'so', 'oa', 'l ', 'fi', 'nc', 'ci', 'am', 'me', 'en', 'nt', 'to', 'tr', 'ro', 'os ', 's c', ' ch', 'che', 'efe', 'es ', 's d', ' de', 'de ', 'e d', 'esa', 'sa ', 'a d', ' da', 'da ', 'a e', ' es', 'est', 'nia', 'ia ', 'a l', ' le', 'let', ' li', 'lit', 'a a', ' al', 'ale', 'lem', 'ema', 'man', 'anh', 'ha ', 'a i', ' it', 'lia', 'esp', 'spa', 'pan', ' e ', 'e e', 'qui', ' as', 'ass', 'ssi', 'sin', 'ina', 'ão ', 'o o', ' o ', 'o a', ' ac', 'cor', 'ord', 'do ', 'o p', ' pa', 'par', 'ara', 'ra ', 'a f', ' fo', 'for', 'rne', 'ece', 'cer', 'er ', 'r p', ' pe', 'ess', 'al ', 'l e', 'e f

In [27]:
for lang in df["labels"].unique():
    subset = df[df["labels"] == lang]["clean_text"]
    X_lang = tfidf.transform(subset)
    nonzero = (X_lang.sum(axis=0) > 0).A1.sum()
    print(lang, "covered features:", nonzero)


pt covered features: 2582
bg covered features: 1940
zh covered features: 491
th covered features: 2404
ru covered features: 1952
pl covered features: 2484
ur covered features: 1825
sw covered features: 2742
tr covered features: 2677
es covered features: 2477
ar covered features: 1726
it covered features: 2550
hi covered features: 2061
de covered features: 2618
el covered features: 2154
nl covered features: 2615
fr covered features: 2569
vi covered features: 2677
en covered features: 2607
ja covered features: 876


In [ ]:
'''
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def build_stratified_vocab(df, n_features=5000, ngram_range=(2,5)):
    langs = df["labels"].unique()
    per_lang = n_features // len(langs)
    vocab = set()

    for lang in langs:
        subset = df[df["labels"] == lang]["clean_text"]
        vec = TfidfVectorizer(analyzer="char", ngram_range=ngram_range)
        X = vec.fit_transform(subset)
        # get feature scores (sum of tf-idf across docs)
        scores = np.asarray(X.sum(axis=0)).ravel()
        indices = scores.argsort()[::-1][:per_lang]
        feats = [f for f, i in vec.vocabulary_.items() if i in indices]
        vocab.update(feats)

    return list(vocab)

# usage
vocab = build_stratified_vocab(df, n_features=5000, ngram_range=(2,5))
print("Final stratified vocab size:", len(vocab))

# build vectorizer with this vocab
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2,5), vocabulary=vocab)
X = tfidf.fit_transform(df["clean_text"])
y = df["labels"]


'''

<center>

# **Modeling**

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# ============================================================================


class LanguageClassifier:


    def __init__(self):
        self.model = LogisticRegression(max_iter=200, solver="lbfgs")
        self.fitted = False


        # mapping labels >> full names
        self.lang_map = {
            "ar": "Arabic", "bg": "Bulgarian", "de": "German", "el": "Modern Greek",
            "en": "English", "es": "Spanish", "fr": "French", "hi": "Hindi",
            "it": "Italian", "ja": "Japanese", "nl": "Dutch", "pl": "Polish",
            "pt": "Portuguese", "ru": "Russian", "sw": "Swahili", "th": "Thai",
            "tr": "Turkish", "ur": "Urdu", "vi": "Vietnamese", "zh": "Chinese"
        }
    # ============================================================================

    def train(self, X, y, test_size=0.2, random_state=42):

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=random_state
        )

        self.model.fit(X_train, y_train)
        self.fitted = True
        y_pred = self.model.predict(X_test)

        print("Classification Report:\n", classification_report(y_test, y_pred))
        print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
        return X_train, X_test, y_train, y_test

    # ============================================================================


    def predict(self, texts, vectorizer):

        if not self.fitted:
            raise ValueError("Model not trained yet. Call train() first.")
        
        X_new = vectorizer.transform(texts)
        preds = self.model.predict(X_new)
        # convert labels → full names
        return [self.lang_map[label] for label in preds]


In [ ]:
clf = LanguageClassifier()
X_train, X_test, y_train, y_test = clf.train(X, y)

sample_texts = ["اهلا عامل ايه؟", "Hello my friend", "こんにちは"]
preds = clf.predict(sample_texts, tfidf.vectorizer)
print(preds)

Classification Report:
               precision    recall  f1-score   support

          ar       1.00      0.99      1.00       700
          bg       0.98      0.98      0.98       700
          de       1.00      1.00      1.00       700
          el       1.00      1.00      1.00       700
          en       0.96      1.00      0.98       700
          es       1.00      1.00      1.00       700
          fr       1.00      1.00      1.00       700
          hi       1.00      1.00      1.00       700
          it       0.99      0.99      0.99       700
          ja       1.00      0.99      1.00       700
          nl       0.98      0.99      0.99       700
          pl       0.99      0.99      0.99       700
          pt       0.99      0.99      0.99       700
          ru       0.98      0.97      0.98       700
          sw       0.99      0.96      0.98       700
          th       1.00      1.00      1.00       700
          tr       1.00      0.98      0.99       700
   

In [32]:
# 1) Load splits
df_train = ds["train"].to_pandas()
df_val   = ds["validation"].to_pandas()
df_test  = ds["test"].to_pandas()

# 2) Preprocess
prep = TextPreprocessor()
df_train["clean_text"] = df_train["text"].apply(prep.clean_text)
df_val["clean_text"]   = df_val["text"].apply(prep.clean_text)
df_test["clean_text"]  = df_test["text"].apply(prep.clean_text)

# 3) TF-IDF (fit train only)
tfidf = TfidfLangVectorizer(max_features=5000, ngram_range=(2,5))
X_train = tfidf.fit_transform(df_train["clean_text"])
y_train = df_train["labels"]

X_val   = tfidf.transform(df_val["clean_text"])
y_val   = df_val["labels"]

X_test  = tfidf.transform(df_test["clean_text"])
y_test  = df_test["labels"]

# 4) Classifier
clf = LanguageClassifier()
clf.model.fit(X_train, y_train)

# 5) Evaluate
print("Validation Report:\n", classification_report(y_val, clf.model.predict(X_val)))
print("Test Report:\n", classification_report(y_test, clf.model.predict(X_test)))

Validation Report:
               precision    recall  f1-score   support

          ar       1.00      0.99      0.99       500
          bg       0.99      0.99      0.99       500
          de       1.00      1.00      1.00       500
          el       1.00      1.00      1.00       500
          en       0.97      1.00      0.99       500
          es       0.99      0.99      0.99       500
          fr       1.00      1.00      1.00       500
          hi       1.00      0.95      0.98       500
          it       1.00      0.99      0.99       500
          ja       1.00      0.99      0.99       500
          nl       0.98      1.00      0.99       500
          pl       1.00      1.00      1.00       500
          pt       0.99      0.99      0.99       500
          ru       0.99      0.99      0.99       500
          sw       0.95      1.00      0.97       500
          th       1.00      0.97      0.98       500
          tr       0.98      1.00      0.99       500
       

In [43]:
clf.fitted = True
sample_texts = ["اهلا بيك كنت حابب اسال عن توبيك mental health"]
preds = clf.predict(sample_texts, tfidf.vectorizer)
print(preds)

['ar']


In [37]:
import joblib

# Save
joblib.dump(clf.model, "language_classifier.pkl")
joblib.dump(tfidf.vectorizer, "tfidf_vectorizer.pkl")

# Load later
loaded_model = joblib.load("language_classifier.pkl")
loaded_vectorizer = joblib.load("tfidf_vectorizer.pkl")


In [ ]:
lang_map = {
    "ar": "Arabic", "bg": "Bulgarian", "de": "German", "el": "Modern Greek",
    "en": "English", "es": "Spanish", "fr": "French", "hi": "Hindi",
    "it": "Italian", "ja": "Japanese", "nl": "Dutch", "pl": "Polish",
    "pt": "Portuguese", "ru": "Russian", "sw": "Swahili", "th": "Thai",
    "tr": "Turkish", "ur": "Urdu", "vi": "Vietnamese", "zh": "Chinese"}

# Example prediction
sample_texts = ["اهلا عامل ايه؟", "Hello my friend", "こんにちは"]
preds = loaded_model.predict(loaded_vectorizer.transform(sample_texts))
preds_full = [lang_map[label] for label in preds]
print(preds_full)


['Arabic', 'English', 'Chinese']


In [44]:
# Example prediction
sample_texts = ["12587*/35&*()"]
preds = loaded_model.predict(loaded_vectorizer.transform(sample_texts))
preds_full = [lang_map[label] for label in preds]
print(preds_full)

['Chinese']
